# AutoMode NeurIPS — Driver for GPU 2 (RTX 5070, 32 GB)

**Role**: Gemma-2-2B for core runs (Tier 0, Tier 4) + ALL ablations (Tier 3).

The 2B model is cheapest, so this GPU absorbs the ablation burden:
- Importance signal ablation (grad_norm / fisher / random / ema_grad)
- LoRA fallback ablation (dyn_full_only)
- Threshold and cadence sweeps

Tier 3 is ~45 runs on 2B only. Combined with Tier 0/4 for 2B, this GPU has
the highest total run count but lowest per-run cost.

In [1]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import torch
print(f'torch {torch.__version__} | cuda={torch.cuda.is_available()} | '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

torch 2.11.0+cu128 | cuda=True | device=NVIDIA GeForce RTX 5090


In [2]:
import os
from huggingface_hub import login
login(os.environ["HF_TOKEN"])


In [3]:
from huggingface_hub import whoami
print(whoami())

In [4]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_switching.py', 'tests/test_training_invariants.py', '-v'],
    cwd='..', capture_output=True, text=True,
)
print(result.stdout); print(result.stderr)
assert result.returncode == 0, 'Unit tests FAILED. Do not proceed to training.'

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /venv/main/bin/python
cachedir: .pytest_cache
rootdir: /workspace/Automode/automode_pkg
configfile: pyproject.toml
plugins: anyio-4.13.0
collecting ... collected 34 items

tests/test_switching.py::TestLayerIdentification::test_double_peft_wrap PASSED [  2%]
tests/test_switching.py::TestLayerIdentification::test_get_transformer_layers_peft_wrapped PASSED [  5%]
tests/test_switching.py::TestLayerIdentification::test_get_transformer_layers_plain PASSED [  8%]
tests/test_switching.py::TestLayerIdentification::test_gets_all_layer_ids PASSED [ 11%]
tests/test_switching.py::TestLayerIdentification::test_layer_ids_sorted_numerically_not_alphabetically PASSED [ 14%]
tests/test_switching.py::TestLayerIdentification::test_non_layer_returns_none PASSED [ 17%]
tests/test_switching.py::TestLayerIdentification::test_peft_wrapped_path PASSED [ 20%]
tests/test

In [5]:
# This GPU claims 2B from Tier 0/4 AND all of Tier 3 (ablations, 2B only)
from automode.grid import build_tier_grid, shard_grid_by_gpu, DEFAULT_GPU_ASSIGNMENT

GPU_RANK = 2

full_grid = build_tier_grid(tiers=[0, 3, 4], output_root='runs')
mine = shard_grid_by_gpu(full_grid, gpu_rank=GPU_RANK, assignment=DEFAULT_GPU_ASSIGNMENT)
for c in mine:
    c.device = 'cuda:0'

gsm8k_only = [c for c in mine if c.train_track == "gsm8k"]
metamath_only = [c for c in mine if c.train_track == "metamath"]

from collections import Counter
print(f'Total grid: {len(full_grid)} | My slice: {len(mine)}')
print(f'By tier:   {dict(Counter(c.tier for c in mine))}')
print(f'By track:  {dict(Counter(c.train_track for c in mine))}')
print(f'By method (top 12):')
for m, n in Counter(c.variant_label() for c in mine).most_common(12):
    print(f'  {n:3d}  {m}')

Total grid: 210 | My slice: 90
By tier:   {0: 36, 3: 30, 4: 24}
By track:  {'metamath': 60, 'gsm8k': 30}
By method (top 12):
    9  automode_u6_t10_grad_norm_r16
    9  automode_u10_t10_grad_norm_r16
    6  full_ft
    6  lora_r16_a32
    6  lora_r32_a64
    6  topk_block_13_19
    6  lisa_k4
    6  adagrad_pct20
    6  adalora_init16_target8
    6  dora_r16
    3  automode_u10_t10_fisher_r16
    3  automode_u10_t10_random_r16


In [6]:
# Smoke test on 2B
from automode.config import preset_automode
from automode.train import run_experiment
from automode.grid import _apply_model_defaults, _apply_track_defaults

smoke = preset_automode(
    u=10, t=10, r=16, alpha=32,
    model_name='google/gemma-2-2b',
    train_track='gsm8k',
    eval_benchmarks=('gsm8k',),
    seed=42, epochs=1,
    output_root='runs/smoke',
    max_train_samples=32, max_eval_samples=50, tier=-1,
)
smoke = _apply_track_defaults(_apply_model_defaults(smoke))
smoke.device = 'cuda:0'
smoke_result = run_experiment(smoke)
assert smoke_result['status'] == 'ok', f'Smoke: {smoke_result}'
print({k: v for k, v in smoke_result.items() if k != 'config'})

[skip] run 8631513abfa5 already complete.
{'run_id': '8631513abfa5', 'method': 'automode', 'variant': 'automode_u10_t10_grad_norm_r16', 'paper_fidelity': 'faithful', 'status': 'ok', 'model_name': 'google/gemma-2-2b', 'train_track': 'gsm8k', 'seed': 42, 'epochs': 1, 'learning_rate': 5e-05, 'train_batch_size': 1, 'grad_accum_steps': 16, 'lora_r': 16, 'lora_alpha': 32, 'lora_target_modules': 'q_proj,v_proj', 'dynamic_updates': 10, 'dynamic_threshold': 10, 'importance_signal': 'grad_norm', 'topk_k': None, 'deep_block_start': None, 'deep_block_end': None, 'adalora_init_r': None, 'adalora_target_r': None, 'lisa_num_layers': None, 'lisa_period': None, 'adagrad_pct': None, 'train_time_sec': 11.555869579315186, 'peak_vram_gb': 6.791213512420654, 'total_opt_steps': 2, 'final_trainable_params': 163160064, 'total_params': 2617536768, 'n_switches': 2, 'eval_gsm8k_maj1': 0.04, 'eval_gsm8k_n_correct': 2, 'eval_gsm8k_n_total': 50}


In [ ]:
from automode.grid import run_grid
import time
CSV_PATH = 'runs/gpu2_results.csv'
t0 = time.time()
results = run_grid(gsm8k_only, csv_path=CSV_PATH, stop_on_error=False, verbose=True)
print(f'\nGPU 2 complete: {len(results)} rows in {(time.time()-t0)/3600:.1f} hr')


[1/30] 55b0f08c734c | google/gemma-2-2b | full_ft | seed=8

[run] 55b0f08c734c | method=full_ft | model=google/gemma-2-2b
      track=gsm8k | seed=8 | epochs=2
      trainable=2,614,341,888 / total=2,614,341,888 (100.000%)



[epoch 1/2] time=2433.9s | last_loss=0.2056 | trainable=2,614,341,888


[epoch 2/2] time=2430.5s | last_loss=0.1113 | trainable=2,614,341,888
[checkpoint] Saved to runs/tier0/55b0f08c734c/checkpoints/final_model



[2/30] badb47649b60 | google/gemma-2-2b | lora_r16_a32 | seed=8

[run] badb47649b60 | method=lora | model=google/gemma-2-2b
      track=gsm8k | seed=8 | epochs=2
      trainable=3,194,880 / total=2,617,536,768 (0.122%)



[epoch 1/2] time=2783.4s | last_loss=0.3454 | trainable=3,194,880


...

[epoch 2/2] time=2189.7s | last_loss=0.2569 | trainable=163,160,064



[11/30] d62530a42295 | google/gemma-2-2b | automode_u10_t10_grad_norm_r16 | seed=25

[run] d62530a42295 | method=automode | model=google/gemma-2-2b
      track=gsm8k | seed=25 | epochs=2
      trainable=3,194,880 / total=2,617,536,768 (0.122%)



[epoch 1/2] time=2323.0s | last_loss=0.4116 | trainable=163,160,064


[epoch 2/2] time=2293.3s | last_loss=0.2404 | trainable=163,160,064


GSM8K eval:  98%|█████████▊| 162/165 [41:04<00:45, 15.20s/it]